In [1]:
"""
Author: Yonglong Tian (yonglong@mit.edu)
Date: May 07, 2020
"""
from __future__ import print_function

import torch
import torch.nn as nn
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print(nn.Linear(10, 5))


class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.1, contrast_mode='all',
                 base_temperature=0.1):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = (torch.device('cuda')
                  if features.is_cuda
                  else torch.device('cpu'))

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # modified to handle edge cases when there is no positive pair
        # for an anchor point. 
        # Edge case e.g.:- 
        # features of shape: [4,1,...]
        # labels:            [0,1,1,2]
        # loss before mean:  [nan, ..., ..., nan] 
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

torch: 2.10.0+cu128
cuda available: True
Linear(in_features=10, out_features=5, bias=True)


In [2]:
import numpy as np
import pandas as pd
import os
import torch
from torch import nn
import torch.nn.functional as F
import random
import math
import time
import sys
import datetime

# ============================================================================
# CONFIGURATION
# ============================================================================
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    RAW_DATA_PATH = '/kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/train_data.csv'
    LABEL_MAPPING_PATH = '/kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/label_mapping.csv'
    MODEL_PATH = '/kaggle/working/pretrain_topsearch10.pth'

# Training parameters
BATCH_SIZE = 32
LEARNING_RATE = 0.1
WEIGHT_DECAY = 1e-4
MOMENTUM = 0.9
EPOCHS = 200
TEMPERATURE = 0.1

# Learning rate schedule
LR_DECAY_EPOCHS = [700, 800, 900]
LR_DECAY_RATE = 0.1
COSINE_SCHEDULE = True
WARMUP = True
WARMUP_EPOCHS = 10
WARMUP_FROM = 0.01

# Model parameters
HIDDEN_SIZE = 256
EMBEDDING_SIZE = 128


print(f"Running on: {'KAGGLE' if IS_KAGGLE else 'LOCAL'}")
print(f"Raw data path: {RAW_DATA_PATH}")

# ============================================================================
# UTILITIES (from original code)
# ============================================================================

class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def adjust_learning_rate(optimizer, epoch):
    """Decay the learning rate based on schedule (following original code)"""
    lr = LEARNING_RATE
    
    if COSINE_SCHEDULE:
        eta_min = lr * (LR_DECAY_RATE ** 3)
        lr = eta_min + (lr - eta_min) * (1 + math.cos(math.pi * epoch / EPOCHS)) / 2
    else:
        steps = np.sum(epoch > np.asarray(LR_DECAY_EPOCHS))
        if steps > 0:
            lr = lr * (LR_DECAY_RATE ** steps)
    
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    return lr


def warmup_learning_rate(optimizer, epoch, batch_id, total_batches):
    """Warmup learning rate (following original code)"""
    if not WARMUP or epoch > WARMUP_EPOCHS:
        return
    
    p = (batch_id + (epoch - 1) * total_batches) / (WARMUP_EPOCHS * total_batches)
    
    if COSINE_SCHEDULE:
        eta_min = LEARNING_RATE * (LR_DECAY_RATE ** 3)
        warmup_to = eta_min + (LEARNING_RATE - eta_min) * (
                1 + math.cos(math.pi * WARMUP_EPOCHS / EPOCHS)) / 2
    else:
        warmup_to = LEARNING_RATE
    
    lr = WARMUP_FROM + p * (warmup_to - WARMUP_FROM)
    
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr


def save_model(model, optimizer, epoch, save_path):
    """Save model checkpoint"""
    print(f'💾 Saving model to {save_path}')
    state = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'model_config': {
            'max_packets': model.max_packets if hasattr(model, 'max_packets') else MAX_PACKETS,
            'hidden_size': HIDDEN_SIZE,
            'embedding_size': EMBEDDING_SIZE,
        }
    }
    torch.save(state, save_path)


# ============================================================================
# DATA PROCESSING
# ============================================================================
SAMPLES_PER_CLASS = 150   # khớp với samples_per_class mặc định của organize_data_by_label
CSV_CHUNKSIZE = 2000


# ============================================================================
# DATA PROCESSING (chunked, capped at n_per_class to avoid loading the whole
# CSV into RAM before capping — replaces the previous read_csv(...) + full
# iterrows() version)
# ============================================================================

def load_and_process_raw_data(raw_data_path, label_mapping_path,
                               n_per_class=SAMPLES_PER_CLASS, chunksize=CSV_CHUNKSIZE,
                               seed=42):
    """Load raw packet data and reshape to (N, max_packets, 3).

    n_per_class: keep at most n_per_class samples per class. Counted while
    reading, so rows beyond the quota are skipped immediately and never held
    in memory — set to None to keep everything (old behavior, can OOM on
    large files).
    chunksize: rows read per pandas chunk, keeps peak memory bounded
    regardless of total file size.
    """
    print("\n" + "="*60)
    print("LOADING RAW PACKET DATA")
    print("="*60)

    if os.path.exists(label_mapping_path):
        label_df = pd.read_csv(label_mapping_path)
        label_names = dict(zip(label_df['label_name'], label_df['label_id']))
        print(f"\n✓ Loaded {len(label_names)} label mappings")
    else:
        print("⚠️  Label mapping not found")
        label_names = {}

    print(f"\n📂 Loading raw data from: {raw_data_path}")
    if n_per_class is not None:
        print(f"  Capping at {n_per_class} samples/class, chunksize={chunksize}")

    all_samples = []
    all_labels = []
    class_counts = {}  # label -> số sample đã giữ
    total_rows_seen = 0
    max_packets = None

    for chunk in pd.read_csv(raw_data_path, header=None, chunksize=chunksize):
        total_rows_seen += len(chunk)

        if max_packets is None:
            total_cols = len(chunk.columns)
            max_packets = total_cols // 4
            print(f"✓ Detected shape: ({max_packets}, 4) per sample")

        for _, row in chunk.iterrows():
            sample = row.values.reshape(max_packets, 4)
            label = int(sample[0, 0])

            if n_per_class is not None:
                count = class_counts.get(label, 0)
                if count >= n_per_class:
                    continue  # đủ quota cho class này rồi, bỏ qua luôn -> nhẹ RAM
                class_counts[label] = count + 1

            sample_features = sample[:, 1:]  # Keep [time, direction, size]
            all_samples.append(sample_features)
            all_labels.append(label)

        print(f"  ...đã quét {total_rows_seen} dòng, đang giữ {len(all_samples)} samples", end='\r')

    print()  # xuống dòng sau progress

    X = np.array(all_samples, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int64)

    print(f"\n✅ Data reshaping complete:")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Unique labels: {len(np.unique(y))}")

    unique_labels, counts = np.unique(y, return_counts=True)
    print(f"\n📊 Label distribution:")
    for label, count in zip(unique_labels, counts):
        label_name = label_names.get(label, f"Label_{label}")
        print(f"  {label_name} (ID={label}): {count} samples")

    return X, y, label_names, max_packets


# ============================================================================
# MODEL ARCHITECTURE (inspired by ResNet structure)
# ============================================================================
# ============================================================================
# MODEL ARCHITECTURE (DF-style)
# ============================================================================

class RawPacketEncoder(nn.Module):
    """
    DF-style Encoder for raw packet sequences
    # Input: (batch, max_packets, 3) - [time, direction, size]
    # Output: (batch, hidden_size)
    # """
    def __init__(self, max_packets, hidden_size=256, embedding_size=128):
            super(RawPacketEncoder, self).__init__()
            self.max_packets = max_packets
            self.hidden_size = hidden_size
            self.embedding_size = embedding_size
    
            kernel_size = 8
            conv_stride = 1
            pool_stride = 4
            pool_size = 8
    
            self.conv1   = nn.Conv1d(3,   32,  kernel_size, stride=conv_stride)
            self.conv1_1 = nn.Conv1d(32,  32,  kernel_size, stride=conv_stride)
            self.conv2   = nn.Conv1d(32,  64,  kernel_size, stride=conv_stride)
            self.conv2_2 = nn.Conv1d(64,  64,  kernel_size, stride=conv_stride)
            self.conv3   = nn.Conv1d(64,  128, kernel_size, stride=conv_stride)
            self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride=conv_stride)
            self.conv4   = nn.Conv1d(128, 256, kernel_size, stride=conv_stride)
            self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride=conv_stride)
    
            self.batch_norm1 = nn.BatchNorm1d(32)
            self.batch_norm2 = nn.BatchNorm1d(64)
            self.batch_norm3 = nn.BatchNorm1d(128)
            self.batch_norm4 = nn.BatchNorm1d(256)
    
            self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
            self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
            self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
            self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
    
            self.dropout1 = nn.Dropout(p=0.1)
            self.dropout2 = nn.Dropout(p=0.1)
            self.dropout3 = nn.Dropout(p=0.1)
            self.dropout4 = nn.Dropout(p=0.1)
    
            # ⚠️ fc phải định nghĩa SAU khi tất cả conv/pool/dropout đã được tạo
            # vì _forward_convs dùng chúng
            with torch.no_grad():
                dummy = torch.zeros(1, 3, max_packets)
                dummy = self._forward_convs(dummy)
                flat_dim = dummy.view(1, -1).shape[1]
            print(f"  ✓ Flat dim after convs: {flat_dim}")
    
            self.fc = nn.Linear(flat_dim, hidden_size)
            self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _forward_convs(self, x):
        """Shared conv forward — dùng cho cả dummy pass và forward thật"""
        # ==== Block 1 (ELU) ====
        x = F.pad(x, (3, 4)); x = F.elu(self.conv1(x))
        x = F.pad(x, (3, 4)); x = F.elu(self.batch_norm1(self.conv1_1(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_1(x)
        x = self.dropout1(x)

        # ==== Block 2 (ReLU) ====
        x = F.pad(x, (3, 4)); x = F.relu(self.conv2(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm2(self.conv2_2(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_2(x)
        x = self.dropout2(x)

        # ==== Block 3 (ReLU) ====
        x = F.pad(x, (3, 4)); x = F.relu(self.conv3(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm3(self.conv3_3(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_3(x)
        x = self.dropout3(x)

        # ==== Block 4 (ReLU) ====
        x = F.pad(x, (3, 4)); x = F.relu(self.conv4(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm4(self.conv4_4(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_4(x)
        x = self.dropout4(x)

        return x

    def forward(self, x):
        # x: (batch, max_packets, 3)
        x = x.transpose(1, 2)      # -> (batch, 3, max_packets)
        x = self._forward_convs(x)
        x = x.view(x.size(0), -1)  # flatten
        feat = self.fc(x)           # -> (batch, hidden_size)
        return feat


#  # model top search 100 tor 
# class RawPacketEncoder(nn.Module):
#     """
#     DF-style Encoder for raw packet sequences
#     Input: (batch, max_packets, 3) - [time, direction, size]
#     Output: (batch, hidden_size)
#     """
#     def __init__(self, max_packets, hidden_size=256, embedding_size=128):
#             super(RawPacketEncoder, self).__init__()
#             self.max_packets = max_packets
#             self.hidden_size = hidden_size
#             self.embedding_size = embedding_size
    
#             kernel_size = 8
#             conv_stride = 1
#             pool_stride = 4
#             pool_size = 8
    
#             self.conv1   = nn.Conv1d(3,   32,  kernel_size, stride=conv_stride)
#             self.conv1_1 = nn.Conv1d(32,  32,  kernel_size, stride=conv_stride)
#             self.conv2   = nn.Conv1d(32,  64,  kernel_size, stride=conv_stride)
#             self.conv2_2 = nn.Conv1d(64,  64,  kernel_size, stride=conv_stride)
#             self.conv3   = nn.Conv1d(64,  128, kernel_size, stride=conv_stride)
#             self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride=conv_stride)
#             self.conv4   = nn.Conv1d(128, 256, kernel_size, stride=conv_stride)
#             self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride=conv_stride)
    
#             self.bn1 = nn.BatchNorm1d(32)
#             self.bn2 = nn.BatchNorm1d(64)
#             self.bn3 = nn.BatchNorm1d(128)
#             self.bn4 = nn.BatchNorm1d(256)
    
#             self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
#             self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
#             self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
#             self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
    
#             self.dropout1 = nn.Dropout(p=0.1)
#             self.dropout2 = nn.Dropout(p=0.1)
#             self.dropout3 = nn.Dropout(p=0.1)
#             self.dropout4 = nn.Dropout(p=0.1)
    
#             # ⚠️ fc phải định nghĩa SAU khi tất cả conv/pool/dropout đã được tạo
#             # vì _forward_convs dùng chúng
#             with torch.no_grad():
#                 dummy = torch.zeros(1, 3, max_packets)
#                 dummy = self._forward_convs(dummy)
#                 flat_dim = dummy.view(1, -1).shape[1]
#             print(f"  ✓ Flat dim after convs: {flat_dim}")
    
#             self.fc = nn.Linear(flat_dim, hidden_size)
#             self._init_weights()

    # def _init_weights(self):
    #     for m in self.modules():
    #         if isinstance(m, (nn.Conv1d, nn.Linear)):
    #             nn.init.xavier_uniform_(m.weight)
    #             if m.bias is not None:
    #                 m.bias.data.zero_()
    #         elif isinstance(m, nn.BatchNorm1d):
    #             nn.init.constant_(m.weight, 1)
    #             nn.init.constant_(m.bias, 0)

    # def _forward_convs(self, x):
    #     """Shared conv forward — dùng cho cả dummy pass và forward thật"""
    #     # ==== Block 1 (ELU) ====
    #     x = F.pad(x, (3, 4)); x = F.elu(self.conv1(x))
    #     x = F.pad(x, (3, 4)); x = F.elu(self.bn1(self.conv1_1(x)))
    #     x = F.pad(x, (3, 4)); x = self.max_pool_1(x)
    #     x = self.dropout1(x)

    #     # ==== Block 2 (ReLU) ====
    #     x = F.pad(x, (3, 4)); x = F.relu(self.conv2(x))
    #     x = F.pad(x, (3, 4)); x = F.relu(self.bn2(self.conv2_2(x)))
    #     x = F.pad(x, (3, 4)); x = self.max_pool_2(x)
    #     x = self.dropout2(x)

    #     # ==== Block 3 (ReLU) ====
    #     x = F.pad(x, (3, 4)); x = F.relu(self.conv3(x))
    #     x = F.pad(x, (3, 4)); x = F.relu(self.bn3(self.conv3_3(x)))
    #     x = F.pad(x, (3, 4)); x = self.max_pool_3(x)
    #     x = self.dropout3(x)

    #     # ==== Block 4 (ReLU) ====
    #     x = F.pad(x, (3, 4)); x = F.relu(self.conv4(x))
    #     x = F.pad(x, (3, 4)); x = F.relu(self.bn4(self.conv4_4(x)))
    #     x = F.pad(x, (3, 4)); x = self.max_pool_4(x)
    #     x = self.dropout4(x)

    #     return x

    # def forward(self, x):
    #     # x: (batch, max_packets, 3)
    #     x = x.transpose(1, 2)      # -> (batch, 3, max_packets)
    #     x = self._forward_convs(x)
    #     x = x.view(x.size(0), -1)  # flatten
    #     feat = self.fc(x)           # -> (batch, hidden_size)
    #     return feat


class SupConPacketNet(nn.Module):
    """
    DF-style backbone + projection head for Supervised Contrastive Learning
    """
    def __init__(self, max_packets, hidden_size=256, embedding_size=128, head='mlp'):
        super(SupConPacketNet, self).__init__()
        self.max_packets = max_packets

        self.encoder = RawPacketEncoder(max_packets, hidden_size, embedding_size)

        # Projection head (DFsimCLR style)
        dim_mlp = hidden_size
        if head == 'linear':
            self.head = nn.Linear(dim_mlp, embedding_size)
        elif head == 'mlp':
            self.head = nn.Sequential(
                nn.Linear(dim_mlp, dim_mlp),
                nn.BatchNorm1d(dim_mlp),
                nn.ReLU(inplace=True),
                nn.Linear(dim_mlp, embedding_size)
            )
        else:
            raise NotImplementedError(f'head not supported: {head}')

    def forward(self, x):
        feat = self.encoder(x)
        feat = F.normalize(self.head(feat), dim=1)
        return feat


# ============================================================================
# DATA AUGMENTATION
# ============================================================================

def packet_augmentation(x):
    """
    Single augmentation function for packet sequences
    Apply random combination of augmentations
    """
    augmented = x.clone()
    
    # 1. Time jitter (randomly shift timestamps)
    if torch.rand(1).item() > 0.2:
        time_col = augmented[:, 0]
        jitter = (torch.rand_like(time_col) - 0.5) * 0.2  # ±10% jitter
        augmented[:, 0] = torch.clamp(time_col + jitter, min=0)
    
    # 2. Size scaling (scale packet sizes)
    if torch.rand(1).item() > 0.2:
        scale = 0.9 + torch.rand(1).item() * 0.2  # 0.9 to 1.1
        augmented[:, 2] = augmented[:, 2] * scale
    
    # 3. Packet dropout (randomly zero out some packets)
    if torch.rand(1).item() > 0.5:
        dropout_rate = 0.1
        mask = torch.rand(x.shape[0]) > dropout_rate
        mask = mask.unsqueeze(1).expand_as(x).to(x.device)
        augmented = augmented * mask.float()
    
    # 4. Gaussian noise
    if torch.rand(1).item() > 0.5:
        noise = torch.randn_like(augmented) * 0.05
        augmented = augmented + noise
    
    # Clean up NaN and Inf
    augmented = torch.nan_to_num(augmented, nan=0.0, posinf=0.0, neginf=0.0)
    
    return augmented


class TwoCropTransform:
    """Create two crops of the same sample (following original code structure)"""
    def __init__(self, transform):
        self.transform = transform
    
    def __call__(self, x):
        return [self.transform(x), self.transform(x)]


# ============================================================================
# DATASET
# ============================================================================

class PacketDataset(torch.utils.data.Dataset):
    """Dataset for packet sequences with two-view augmentation"""
    def __init__(self, data_list, transform=None):
        self.transform = transform
        self.samples = []
        self.labels = []
        
        for label_idx, samples in enumerate(data_list):
            for sample in samples:
                if isinstance(sample, np.ndarray):
                    sample = torch.from_numpy(sample.astype(np.float32))
                self.samples.append(sample)
                self.labels.append(label_idx)
        
        print(f"  Dataset: {len(self.samples)} samples, {len(data_list)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        label = self.labels[idx]
        
        if self.transform is not None:
            views = self.transform(sample)
            return views, label
        else:
            return sample, label


# ============================================================================
# TRAINING
# ============================================================================

def train(train_loader, model, criterion, optimizer, epoch):
    """One epoch training (following original code structure)"""
    model.train()
    
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    
    end = time.time()
    
    for idx, (images, labels) in enumerate(train_loader):
        data_time.update(time.time() - end)
        
        # images is a list of [view1, view2]
        images = torch.cat([images[0], images[1]], dim=0)
        
        if torch.cuda.is_available():
            images = images.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)
        
        bsz = labels.shape[0]
        
        # Warmup learning rate
        warmup_learning_rate(optimizer, epoch, idx, len(train_loader))
        
        # Forward
        features = model(images)
        f1, f2 = torch.split(features, [bsz, bsz], dim=0)
        features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
        
        # Compute loss
        loss = criterion(features, labels)
        
        # Check for NaN
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  ⚠️  WARNING: NaN/Inf loss at batch {idx}")
            continue
        
        # Update
        losses.update(loss.item(), bsz)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Measure time
        batch_time.update(time.time() - end)
        end = time.time()
        

    return losses.avg


# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

def main():
    """Main training function"""
    print("\n" + "="*60)
    print("RAW PACKET EMBEDDING WITH SUPCON")
    print("="*60)
    
    # Load data
    X, y, label_names, max_packets = load_and_process_raw_data(
        RAW_DATA_PATH, 
        LABEL_MAPPING_PATH,    n_per_class=SAMPLES_PER_CLASS, chunksize=CSV_CHUNKSIZE,
                               seed=42
    )

    
    # Organize by label
    data_list, filtered_labels = organize_data_by_label(X, y)
    
    if len(data_list) < 2:
        print("\n❌ ERROR: Need at least 2 classes!")
        return False
    
    # Split train/test
    random.shuffle(data_list)
    n_train = max(1, int(len(data_list)))
    
    train_data_list = data_list[:n_train]
    test_data_list = data_list[n_train:] if n_train < len(data_list) else train_data_list
    
    print(f"\n✓ Data split:")
    print(f"  Training classes: {len(train_data_list)}")
    print(f"  Testing classes: {len(test_data_list)}")
    
    # Create datasets
    print(f"\n📦 Creating datasets...")
    transform = TwoCropTransform(packet_augmentation)
    train_dataset = PacketDataset(train_data_list, transform=transform)
    
    # Adjust batch size
    actual_batch_size = min(BATCH_SIZE, len(train_dataset))
    
    # Create dataloader
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=actual_batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=(len(train_dataset) > actual_batch_size)
    )
    
    if len(train_loader) == 0:
        print("\n❌ ERROR: train_loader is empty!")
        return False
    
    print(f"✓ Train batches: {len(train_loader)}")
    
    # Build model
    print(f"\n🏗️  Building model...")
    model = SupConPacketNet(
        max_packets=max_packets,
        hidden_size=HIDDEN_SIZE,
        embedding_size=EMBEDDING_SIZE,
        head='mlp'
    )
    criterion = SupConLoss(temperature=TEMPERATURE)
    
    if torch.cuda.is_available():
        model = model.cuda()
        criterion = criterion.cuda()
        print("✓ Using CUDA")
    else:
        print("✓ Using CPU")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"✓ Model parameters: {total_params:,}")
    
    # Optimizer (following original code)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )
    
    print(f"\n🚀 Starting training...")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Batch size: {actual_batch_size}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print(f"  Temperature: {TEMPERATURE}")
    print(f"  Warmup: {WARMUP_EPOCHS} epochs" if WARMUP else "  No warmup")
    print(f"  LR schedule: {'Cosine' if COSINE_SCHEDULE else 'Step'}")
    
    # Create save directory
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    
    # Training loop
    for epoch in range(1, EPOCHS + 1):
        # Adjust learning rate
        adjust_learning_rate(optimizer, epoch)
        
        # Train
        time1 = time.time()
        loss = train(train_loader, model, criterion, optimizer, epoch)
        time2 = time.time()
        if epoch % 50 == 0:
            print(f'epoch {epoch}, total time {time2 - time1:.2f}, loss {loss}' )
        
    
    # Save final model
    save_model(model, optimizer, EPOCHS, MODEL_PATH)
    
    print(f"\n{'='*60}")
    print("✅ TRAINING COMPLETED")
    print(f"{'='*60}")
    print(f"Model saved to: {MODEL_PATH}")
    
    return True


# if __name__ == '__main__':
#     success = main()
    
#     if not success:
#         print("\n❌ Training failed")
#         sys.exit(1)

Running on: KAGGLE
Raw data path: /kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/train_data.csv


In [3]:
# ============================================================================
# SUBSAMPLING
# ============================================================================

def subsample_per_class(X, y, n_per_class=100, seed=42):
    """Lấy tối đa n_per_class samples cho mỗi class để giảm tải khi train."""
    rng = np.random.RandomState(seed)
    selected_idx = []

    for label in np.unique(y):
        idx = np.where(y == label)[0]
        if len(idx) > n_per_class:
            idx = rng.choice(idx, size=n_per_class, replace=False)
        selected_idx.append(idx)

    selected_idx = np.concatenate(selected_idx)
    rng.shuffle(selected_idx)

    return X[selected_idx], y[selected_idx]

In [4]:
import numpy as np
import pandas as pd
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import classification_report, confusion_matrix

# ============================================================================
# CONFIGURATION
# ============================================================================
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    EMBEDDING_DATA_PATH = '/kaggle/input/datasets/linhhphng/aol-csv/AOL_csv/masque/train_data.csv'
    TEST_DATA_PATH = '/kaggle/input/datasets/linhhphng/aol-csv/AOL_csv/masque/test_data.csv'
    LABEL_MAPPING_PATH = '/kaggle/input/datasets/linhhphng/aol-csv/AOL_csv/masque/label_mapping.csv'
    MODEL_PATH = '/kaggle/input/datasets/linhhphng/aol-csv/AOL_csv/masque/pretrain_masque_AOL.pth'
    OUTPUT_FOLDER = '/kaggle/working/output'

# MLP classifier config
MLP_HIDDEN = 256      # matches encoder hidden_size for consistency with the correction head
MLP_EPOCHS = 100
MLP_BATCH_SIZE = 64
MLP_LR = 1e-3


def load_pretrained_embedding_model(model_path):
    print(f"\n📥 Loading pretrained embedding model from: {model_path}")
    
    if not os.path.exists(model_path):
        print(f"  ❌ Model file not found!")
        return None
    
    checkpoint = torch.load(model_path, map_location='cpu')
    config = checkpoint.get('model_config', {})
    max_packets = config.get('max_packets', 25000)
    hidden_size = config.get('hidden_size', 256)
    embedding_size = config.get('embedding_size', 128)
    
    model = SupConPacketNet(max_packets=max_packets, hidden_size=hidden_size, 
                           embedding_size=embedding_size, head='mlp')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    
    print(f"  ✓ Model loaded on {device} (epoch {checkpoint.get('epoch', '?')})")
    return model

# df model trước ph
def extract_embeddings_from_raw(model, X_raw, batch_size=32):
    """Extract features after encoder fc, before projection head — shape: (batch, hidden_size)"""
    encoder = model.encoder
    encoder.eval()
    device = next(model.parameters()).device
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(X_raw), batch_size):
            batch = X_raw[i:i + batch_size]
            batch_tensor = torch.FloatTensor(batch).to(device)
            
            # encoder.forward() = _forward_convs + flatten + fc -> (batch, hidden_size)
            feat = encoder(batch_tensor)
            embeddings.append(feat.cpu().numpy())

    return np.vstack(embeddings)


# ============================================================================
# MLP CLASSIFIER
# ============================================================================

class MLPClassifier(nn.Module):
    """Same shape as the correction-stage head: Linear -> BN -> ReLU -> Linear"""
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)


def train_mlp_classifier(X_train_emb, y_train, num_classes, device,
                          hidden_dim=MLP_HIDDEN, epochs=MLP_EPOCHS,
                          batch_size=MLP_BATCH_SIZE, lr=MLP_LR):
    model = MLPClassifier(X_train_emb.shape[1], hidden_dim, num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    dataset = TensorDataset(
        torch.FloatTensor(X_train_emb),
        torch.LongTensor(y_train)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        drop_last=(len(dataset) > batch_size))

    model.train()
    for epoch in range(epochs):
        epoch_loss, n_batches = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch + 1}/{epochs}, loss {epoch_loss / n_batches:.4f}")

    return model


def predict_mlp_classifier(model, X_emb, device, batch_size=MLP_BATCH_SIZE):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_emb), batch_size):
            batch = torch.FloatTensor(X_emb[i:i + batch_size]).to(device)
            logits = model(batch)
            preds.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)


# ============================================================================
# DATA LOADING
# ============================================================================
# ============================================================================
# CLASS SELECTION
# ============================================================================
# Dataset has 100 classes total but only work with a subset — edit this list
# to choose which original class ids to load/train/test/correct on.
SELECTED_CLASS_IDS = list(range(100))   # e.g. first 50 of 100 classes


def load_raw_packet_data(data_path, label_mapping_path, data_type='data',
                          n_per_class=None, chunksize=2000, seed=42,
                          selected_classes=None):
    """
    n_per_class: nếu set, chỉ giữ tối đa n_per_class samples/class, đọc theo chunk
    để tránh load hết file vào RAM.
    selected_classes: nếu set, chỉ giữ các sample có label nằm trong tập này —
    các class khác bị bỏ qua ngay khi đọc, không load vào RAM.
    """
    print(f"\n{'='*60}")
    print(f"LOADING {data_type.upper()} DATA")
    print(f"{'='*60}")

    if os.path.exists(label_mapping_path):
        label_df = pd.read_csv(label_mapping_path)

        name_col_candidates = ['label_name', 'name', 'class_name', 'label']
        id_col_candidates = ['label_id', 'id', 'class_id']

        name_col = next((c for c in name_col_candidates if c in label_df.columns), None)
        id_col = next((c for c in id_col_candidates if c in label_df.columns), None)

        if name_col is None or id_col is None:
            print(f"❌ Could not detect label/id columns. Found columns: {list(label_df.columns)}")
            label_names = {}
        else:
            label_names = dict(zip(label_df[name_col], label_df[id_col]))
            print(f"✓ Loaded {len(label_names)} label mappings (columns: '{name_col}', '{id_col}')")
    else:
        label_names = {}

    if not os.path.exists(data_path):
        print(f"❌ Data file not found: {data_path}")
        return None, None, label_names

    selected_set = set(selected_classes) if selected_classes is not None else None
    if selected_set is not None:
        print(f"  Restricting to {len(selected_set)} selected classes")

    rng = np.random.RandomState(seed)

    all_samples = []
    all_labels = []
    class_counts = {}  # label -> số sample đã giữ
    total_rows_seen = 0

    for chunk in pd.read_csv(data_path, header=None, chunksize=chunksize):
        total_rows_seen += len(chunk)
        total_cols = len(chunk.columns)
        max_packets = total_cols // 4

        for _, row in chunk.iterrows():
            sample = row.values.reshape(max_packets, 4)
            label = int(sample[0, 0])

            if selected_set is not None and label not in selected_set:
                continue  # class không được chọn — bỏ qua, không giữ vào RAM

            if n_per_class is not None:
                count = class_counts.get(label, 0)
                if count >= n_per_class:
                    continue  # đủ quota cho class này rồi, bỏ qua luôn -> nhẹ RAM
                class_counts[label] = count + 1

            sample_features = sample[:, 1:]
            all_samples.append(sample_features)
            all_labels.append(label)

        print(f"  ...đã quét {total_rows_seen} dòng, đang giữ {len(all_samples)} samples", end='\r')

    print()  # xuống dòng sau progress

    X = np.array(all_samples, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int64)

    print(f"✓ X: {X.shape}, y: {y.shape}, labels: {len(np.unique(y))}")
    return X, y, label_names


def remap_labels(y, selected_classes):
    """Remap arbitrary original class ids to a contiguous 0..K-1 range.
    Same mapping must be used for train and test — pass the same selected_classes."""
    sorted_ids = sorted(selected_classes)
    old_to_new = {old: new for new, old in enumerate(sorted_ids)}
    y_remapped = np.array([old_to_new[label] for label in y], dtype=np.int64)
    return y_remapped, old_to_new


# ============================================================================
# MAIN
# ============================================================================

print("\n" + "="*60)
print("MLP CLASSIFIER WITH EMBEDDINGS")
print("="*60)

# Load model
emb_model = load_pretrained_embedding_model(MODEL_PATH)

device = next(emb_model.parameters()).device

# Load data — chỉ giữ SELECTED_CLASS_IDS, 200 samples/class, đọc theo chunk để nhẹ RAM
X_train_raw, y_train, label_names = load_raw_packet_data(
    EMBEDDING_DATA_PATH, LABEL_MAPPING_PATH, 'TRAIN',
    n_per_class=200, selected_classes=SELECTED_CLASS_IDS
)
X_test_raw, y_test, _ = load_raw_packet_data(
    TEST_DATA_PATH, LABEL_MAPPING_PATH, 'TEST',
    selected_classes=SELECTED_CLASS_IDS
)

# Remap to contiguous ids 0..K-1 using the same mapping for train and test —
# required so NUM_CLASSES / F.one_hot downstream (classifier, kreciprocal) stay valid
y_train, old_to_new = remap_labels(y_train, SELECTED_CLASS_IDS)
y_test, _ = remap_labels(y_test, SELECTED_CLASS_IDS)
new_to_old = {new: old for old, new in old_to_new.items()}

# label_names is keyed by original ids from label_mapping.csv — reindex to new ids
label_names = {new_id: label_names.get(old_id, f'Class_{old_id}')
               for new_id, old_id in new_to_old.items()}

print(f"\n✓ Working with {len(SELECTED_CLASS_IDS)} classes (remapped to 0..{len(SELECTED_CLASS_IDS)-1})")

# Extract embeddings
print("\n🔄 Extracting embeddings...")
X_train_emb = extract_embeddings_from_raw(emb_model, X_train_raw)
X_test_emb = extract_embeddings_from_raw(emb_model, X_test_raw)

# Train MLP classifier
print(f"\n{'='*60}")
print("TRAINING MLP CLASSIFIER")
print(f"{'='*60}")

num_classes = len(np.unique(np.concatenate([y_train, y_test])))
print("Training...")
clf = train_mlp_classifier(X_train_emb, y_train, num_classes, device)

# Predict
print("\nPredicting on test set...")
y_pred = predict_mlp_classifier(clf, X_test_emb, device)

# Get label names for classification report
unique_labels = np.unique(np.concatenate([y_train, y_test]))
target_names = [label_names.get(label, f'Class_{label}') for label in unique_labels]

# Print classification report
print(f"\n{'='*60}")
print("CLASSIFICATION REPORT")
print(f"{'='*60}\n")

report = classification_report(
    y_test,
    y_pred,
    target_names=[str(x) for x in target_names],
    digits=4,
    zero_division=0
)
print(report)

# Print confusion matrix info
print(f"\n{'='*60}")
print("CONFUSION MATRIX SUMMARY")
print(f"{'='*60}\n")

cm = confusion_matrix(y_test, y_pred, labels=unique_labels)

for i, label in enumerate(unique_labels):
    label_name = label_names.get(label, f'Class_{label}')
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy = correct / total if total > 0 else 0
    print(f"{label_name:30s} - Correct: {correct:5d}/{total:5d} ({accuracy:.2%})")

# Save detailed results
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Save classification report to file
report_path = os.path.join(OUTPUT_FOLDER, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("CLASSIFICATION REPORT - MLP\n")
    f.write("="*60 + "\n\n")
    f.write(report)

print(f"\n✓ Classification report saved to: {report_path}")


MLP CLASSIFIER WITH EMBEDDINGS

📥 Loading pretrained embedding model from: /kaggle/input/datasets/linhhphng/aol-csv/AOL_csv/masque/pretrain_masque_AOL.pth
  ✓ Flat dim after convs: 10240
  ✓ Model loaded on cuda (epoch 800)

LOADING TRAIN DATA
✓ Loaded 50 label mappings (columns: 'label_name', 'label_id')
  Restricting to 100 selected classes
  ...đã quét 15000 dòng, đang giữ 10000 samples
✓ X: (10000, 10000, 3), y: (10000,), labels: 50

LOADING TEST DATA
✓ Loaded 50 label mappings (columns: 'label_name', 'label_id')
  Restricting to 100 selected classes
  ...đã quét 5000 dòng, đang giữ 5000 samples
✓ X: (5000, 10000, 3), y: (5000,), labels: 50

✓ Working with 100 classes (remapped to 0..99)

🔄 Extracting embeddings...

TRAINING MLP CLASSIFIER
Training...
epoch 10/100, loss 0.0042
epoch 20/100, loss 0.0020
epoch 30/100, loss 0.0036
epoch 40/100, loss 0.0005
epoch 50/100, loss 0.0003
epoch 60/100, loss 0.0011
epoch 70/100, loss 0.0002
epoch 80/100, loss 0.0003
epoch 90/100, loss 0.0003

In [5]:
# ============================================================================
# ITERATIVE HARD-EXAMPLE FINE-TUNING — ENCODER FROZEN, CLASSIFIER-ONLY
# Encoder and projection head are fully frozen (no fine-tuning, no gradient).
# Only the MLP classifier head is trained. After each round, the classifier
# predicts on the full training set; samples whose prediction disagrees with
# the ground-truth label are treated as hard examples and get a higher loss
# weight in the next round. Correctly classified samples keep a normal loss
# weight to avoid overfitting to the hard subset.
#
# Removed compared to the previous version: k-reciprocal relabeling, noisy-
# label correction, forget-rate scheduling, XBM/triplet loss, and any
# encoder fine-tuning.
#
# Added: a center-loss regularization term alongside the weighted CE loss.
# Each class keeps a running centroid in the classifier's hidden feature
# space; every sample is pulled toward its class centroid, weighted by the
# same sample_weight as the CE loss — so hard (misclassified) samples get
# pulled toward their correct class centroid much harder than already-
# correct samples, directly pushing them toward the right cluster instead
# of only reweighting their CE loss.
# ============================================================================
import torch.optim as optim

device = next(emb_model.parameters()).device
encoder = emb_model.encoder
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

NUM_CLASSES = len(np.unique(y_train))
CLS_HIDDEN = HIDDEN_SIZE  # 256, encoder output dim

# ---- configuration ----
HF_ROUNDS = 10              # number of hard-example reweighting rounds
HF_EPOCHS_PER_ROUND = 3     # classifier epochs trained per round — kept low so train_acc
                             # does not saturate before the next hard-example evaluation
HF_BATCH_SIZE = 64
HF_LR = 1e-3
WEIGHT_DECAY = 1e-4          # L2 regularization, slows down train-set memorization
DROPOUT_P = 0.3               # dropout in the classifier head, same purpose
HARD_WEIGHT = 5.0           # loss weight for samples misclassified in the previous round
NORMAL_WEIGHT = 1.0         # loss weight for correctly classified samples
CENTER_LOSS_WEIGHT = 0.05   # weight of the center-loss regularization term
CENTER_MOMENTUM = 0.9       # EMA momentum for updating class centroids


class MLPClassifierHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout_p=DROPOUT_P):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        feat = self.feature_extractor(x)
        logits = self.fc(feat)
        return logits, feat


@torch.no_grad()
def extract_all_encoder_features(X_raw, batch_size=HF_BATCH_SIZE):
    """Encoder is frozen, so its output is fixed — compute once and reuse every round."""
    encoder.eval()
    feats = []
    for i in range(0, len(X_raw), batch_size):
        batch = torch.FloatTensor(X_raw[i:i + batch_size]).to(device)
        feats.append(encoder(batch))
    return torch.cat(feats, dim=0)


@torch.no_grad()
def predict_from_features(classifier, feats, batch_size=HF_BATCH_SIZE):
    classifier.eval()
    preds = []
    for i in range(0, feats.size(0), batch_size):
        logits, _ = classifier(feats[i:i + batch_size])
        preds.append(logits.argmax(dim=1))
    return torch.cat(preds, dim=0)


@torch.no_grad()
def compute_class_centroids(classifier, feats, labels, num_classes, batch_size=HF_BATCH_SIZE):
    """Initial centroids from the classifier's current (possibly untrained) features."""
    classifier.eval()
    all_feat = []
    for i in range(0, feats.size(0), batch_size):
        _, feat_b = classifier(feats[i:i + batch_size])
        all_feat.append(feat_b)
    all_feat = torch.cat(all_feat, dim=0)

    centroids = torch.zeros(num_classes, all_feat.size(1), device=feats.device)
    for c in range(num_classes):
        mask = labels == c
        if mask.sum() > 0:
            centroids[c] = all_feat[mask].mean(dim=0)
    classifier.train()
    return centroids


def update_centroids(centroids, feats, labels, momentum):
    """EMA update of each class centroid using the batch's own hidden features."""
    centroids = centroids.clone()
    for c in labels.unique():
        mask = labels == c
        class_mean = feats[mask].mean(dim=0)
        centroids[c] = momentum * centroids[c] + (1 - momentum) * class_mean
    return centroids


def train_classifier_weighted(classifier, optimizer, feats, labels, weights,
                               epochs, batch_size, centroids=None,
                               center_weight=0.0, center_momentum=CENTER_MOMENTUM):
    dataset = torch.utils.data.TensorDataset(feats, labels, weights)
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        drop_last=(len(dataset) > batch_size)
    )
    classifier.train()
    for epoch in range(epochs):
        epoch_loss, epoch_ce, epoch_reg, n_batches = 0.0, 0.0, 0.0, 0
        for feat_b, label_b, weight_b in loader:
            logits, hidden_feat = classifier(feat_b)
            per_sample_ce = F.cross_entropy(logits, label_b, reduction='none')
            ce_loss = (per_sample_ce * weight_b).mean()

            if centroids is not None and center_weight > 0:
                per_sample_center = ((hidden_feat - centroids[label_b]) ** 2).sum(dim=1)
                reg_loss = (per_sample_center * weight_b).mean()
                loss = ce_loss + center_weight * reg_loss
            else:
                reg_loss = torch.tensor(0.0, device=feat_b.device)
                loss = ce_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if centroids is not None and center_weight > 0:
                centroids = update_centroids(centroids, hidden_feat.detach(), label_b, center_momentum)

            epoch_loss += loss.item()
            epoch_ce += ce_loss.item()
            epoch_reg += reg_loss.item()
            n_batches += 1
    return epoch_loss / n_batches, epoch_ce / n_batches, epoch_reg / n_batches, centroids


def run_iterative_hard_example_finetuning(X_raw, y):
    print("\n" + "=" * 60)
    print("ITERATIVE HARD-EXAMPLE FINE-TUNING (encoder frozen, classifier-only)")
    print("=" * 60)

    N = len(y)
    y_tensor = torch.LongTensor(y).to(device)
    enc_feats_all = extract_all_encoder_features(X_raw)

    classifier = MLPClassifierHead(CLS_HIDDEN, CLS_HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(classifier.parameters(), lr=HF_LR, weight_decay=WEIGHT_DECAY)

    sample_weights = torch.full((N,), NORMAL_WEIGHT, device=device)
    prev_n_correct = None
    centroids = compute_class_centroids(classifier, enc_feats_all, y_tensor, NUM_CLASSES)

    for round_idx in range(1, HF_ROUNDS + 1):
        avg_loss, avg_ce, avg_reg, centroids = train_classifier_weighted(
            classifier, optimizer, enc_feats_all, y_tensor, sample_weights,
            epochs=HF_EPOCHS_PER_ROUND, batch_size=HF_BATCH_SIZE,
            centroids=centroids, center_weight=CENTER_LOSS_WEIGHT
        )

        # Evaluate on the full training set to find hard examples for the next round
        preds = predict_from_features(classifier, enc_feats_all)
        correct_mask = (preds == y_tensor)
        n_correct = int(correct_mask.sum().item())
        n_hard = N - n_correct
        train_acc = n_correct / N

        sample_weights = torch.where(
            correct_mask,
            torch.full((N,), NORMAL_WEIGHT, device=device),
            torch.full((N,), HARD_WEIGHT, device=device),
        )

        delta_str = "" if prev_n_correct is None else f" (+{n_correct - prev_n_correct})"
        prev_n_correct = n_correct

        print(f"round {round_idx}/{HF_ROUNDS} | loss {avg_loss:.4f} (ce {avg_ce:.4f}, center {avg_reg:.4f}) | "
              f"correct {n_correct}/{N}{delta_str} | train_acc {train_acc:.4f} | "
              f"hard_examples {n_hard}/{N} ({n_hard / N:.2%})")

    return classifier, enc_feats_all


classifier, X_train_enc_feats = run_iterative_hard_example_finetuning(X_train_raw, y_train)


# ============================================================================
# EVALUATE: HARD-EXAMPLE FINE-TUNED CLASSIFIER vs UNIFORM-WEIGHT BASELINE
# Both are trained on the same fixed embeddings for the same total number of
# epochs (HF_ROUNDS * HF_EPOCHS_PER_ROUND), so the only difference is whether
# misclassified samples get an increased loss weight over successive rounds.
# ============================================================================

X_test_emb = extract_embeddings_from_raw(emb_model, X_test_raw)
X_test_enc_feats = torch.FloatTensor(X_test_emb).to(device)

# Baseline: same training budget, uniform weight throughout (no reweighting)
baseline_classifier = MLPClassifierHead(CLS_HIDDEN, CLS_HIDDEN, NUM_CLASSES).to(device)
baseline_optimizer = optim.Adam(baseline_classifier.parameters(), lr=HF_LR, weight_decay=WEIGHT_DECAY)
uniform_weights = torch.full((len(y_train),), NORMAL_WEIGHT, device=device)
train_classifier_weighted(
    baseline_classifier, baseline_optimizer, X_train_enc_feats,
    torch.LongTensor(y_train).to(device), uniform_weights,
    epochs=HF_ROUNDS * HF_EPOCHS_PER_ROUND, batch_size=HF_BATCH_SIZE
    # no centroids passed — baseline stays plain weighted-CE, center_weight defaults to 0
)

y_pred_hard = predict_from_features(classifier, X_test_enc_feats).cpu().numpy()
y_pred_baseline = predict_from_features(baseline_classifier, X_test_enc_feats).cpu().numpy()

unique_labels = np.unique(np.concatenate([y_train, y_test]))
target_names = [label_names.get(label, f'Class_{label}') for label in unique_labels]


def print_and_save_report(y_true, y_pred, tag, filename):
    print(f"\n{'='*60}")
    print(f"CLASSIFICATION REPORT — {tag}")
    print(f"{'='*60}\n")
    report = classification_report(
        y_true, y_pred,
        target_names=[str(x) for x in target_names],
        digits=4, zero_division=0
    )
    print(report)

    report_path = os.path.join(OUTPUT_FOLDER, filename)
    with open(report_path, 'w') as f:
        f.write(f"CLASSIFICATION REPORT — {tag}\n")
        f.write("="*60 + "\n\n")
        f.write(report)
    print(f"\nReport saved to: {report_path}")
    return report


report_hard = print_and_save_report(
    y_test, y_pred_hard, 'ITERATIVE HARD-EXAMPLE FINE-TUNING',
    'classification_report_hard_example.txt'
)
report_baseline = print_and_save_report(
    y_test, y_pred_baseline, 'UNIFORM-WEIGHT BASELINE',
    'classification_report_baseline.txt'
)

acc_hard = (y_pred_hard == y_test).mean()
acc_baseline = (y_pred_baseline == y_test).mean()
print(f"\n{'='*60}")
print(f"Overall accuracy — uniform-weight baseline : {acc_baseline:.4f}")
print(f"Overall accuracy — hard-example fine-tuning: {acc_hard:.4f}")
print(f"{'='*60}")


ITERATIVE HARD-EXAMPLE FINE-TUNING (encoder frozen, classifier-only)
round 1/10 | loss 1.0629 (ce 0.2237, center 16.7848) | correct 9969/10000 | train_acc 0.9969 | hard_examples 31/10000 (0.31%)
round 2/10 | loss 0.6531 (ce 0.1804, center 9.4546) | correct 9979/10000 (+10) | train_acc 0.9979 | hard_examples 21/10000 (0.21%)
round 3/10 | loss 0.4693 (ce 0.1311, center 6.7655) | correct 9983/10000 (+4) | train_acc 0.9983 | hard_examples 17/10000 (0.17%)
round 4/10 | loss 0.3686 (ce 0.1009, center 5.3541) | correct 9985/10000 (+2) | train_acc 0.9985 | hard_examples 15/10000 (0.15%)
round 5/10 | loss 0.3056 (ce 0.0861, center 4.3909) | correct 9989/10000 (+4) | train_acc 0.9989 | hard_examples 11/10000 (0.11%)
round 6/10 | loss 0.2572 (ce 0.0721, center 3.7016) | correct 9986/10000 (+-3) | train_acc 0.9986 | hard_examples 14/10000 (0.14%)
round 7/10 | loss 0.2279 (ce 0.0674, center 3.2088) | correct 9986/10000 (+0) | train_acc 0.9986 | hard_examples 14/10000 (0.14%)
round 8/10 | loss 0.20

In [6]:
# ============================================================================
# ITERATIVE HARD-EXAMPLE FINE-TUNING — ENCODER FROZEN, CLASSIFIER-ONLY
# Encoder and projection head are fully frozen (no fine-tuning, no gradient).
# Only the MLP classifier head is trained. After each round, the classifier
# predicts on the full training set; samples whose prediction disagrees with
# the ground-truth label are treated as hard examples and get a higher loss
# weight in the next round. Correctly classified samples keep a normal loss
# weight to avoid overfitting to the hard subset.
#
# Regularizer: class-conditional MK-MMD alongside the weighted CE loss.
# For every class that has hard samples in the current batch, the MK-MMD
# distance is measured between those hard samples' hidden features and the
# hidden features of correctly classified samples of the same label. The
# correct-sample bank is rebuilt at the start of every epoch from the
# classifier's current features (detached, so gradients only move the hard
# samples toward their own class's correct samples).
# ============================================================================
import torch.optim as optim
from typing import Optional, Sequence

device = next(emb_model.parameters()).device
encoder = emb_model.encoder
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

NUM_CLASSES = len(np.unique(y_train))
CLS_HIDDEN = HIDDEN_SIZE  # 256, encoder output dim

# ---- configuration ----
HF_ROUNDS = 10              # number of hard-example reweighting rounds
HF_EPOCHS_PER_ROUND = 3     # classifier epochs trained per round — kept low so train_acc
                             # does not saturate before the next hard-example evaluation
HF_BATCH_SIZE = 64
HF_LR = 1e-3
WEIGHT_DECAY = 1e-4          # L2 regularization, slows down train-set memorization
DROPOUT_P = 0.3               # dropout in the classifier head, same purpose
HARD_WEIGHT = 5.0           # loss weight for samples misclassified in the previous round
NORMAL_WEIGHT = 1.0         # loss weight for correctly classified samples
MMD_WEIGHT = 0.1            # weight of the MK-MMD regularization term
MMD_TARGET_SIZE = 32        # max correct same-label samples used as the MMD target per class
MMD_ALPHAS = [0.001, 0.01, 0.1, 1.0]  # Gaussian kernel bandwidths (alpha = 1 / (2 sigma^2))


# ---- MK-MMD (from the provided implementation) ----
# Changes vs. the provided version: the index-matrix cache is a dict keyed by
# (n_s, n_t, device) because the hard-sample count varies per class per batch,
# and the non-linear index matrix is built without python loops (same values).
class GaussianKernel(nn.Module):
    """Gaussian Kernel for MK-MMD computation"""
    def __init__(self, alpha: float = 1.0):
        super(GaussianKernel, self).__init__()
        self.alpha = alpha

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        n = x.size(0)
        x_norm = (x ** 2).sum(dim=1).view(n, 1)
        dist = x_norm + x_norm.view(1, n) - 2.0 * torch.mm(x, x.transpose(0, 1))
        return torch.exp(-self.alpha * dist)


class MultipleKernelMaximumMeanDiscrepancy(nn.Module):
    def __init__(self, kernels: Sequence[nn.Module], linear: Optional[bool] = False):
        super(MultipleKernelMaximumMeanDiscrepancy, self).__init__()
        self.kernels = kernels
        self.linear = linear
        self._index_cache = {}

    def forward(self, z_s: torch.Tensor, z_t: torch.Tensor) -> torch.Tensor:
        if z_s.dim() > 2:
            z_s = z_s.view(z_s.size(0), -1)
        if z_t.dim() > 2:
            z_t = z_t.view(z_t.size(0), -1)

        features = torch.cat([z_s, z_t], dim=0)
        n_s = int(z_s.size(0))
        n_t = int(z_t.size(0))

        key = (n_s, n_t, str(z_s.device))
        if key not in self._index_cache:
            self._index_cache[key] = self._update_index_matrix(n_s, n_t, self.linear).to(z_s.device)
        index_matrix = self._index_cache[key]

        kernel_matrix = sum([kernel(features) for kernel in self.kernels])
        return (kernel_matrix * index_matrix).sum()

    def _update_index_matrix(self, n_s: int, n_t: int, linear: Optional[bool] = True) -> torch.Tensor:
        total_size = n_s + n_t
        index_matrix = torch.zeros(total_size, total_size)

        if linear:
            for i in range(n_s):
                j = (i + 1) % n_s
                index_matrix[i, j] = 1. / float(n_s)

            for i in range(n_t):
                j = (i + 1) % n_t
                index_matrix[n_s + i, n_s + j] = 1. / float(n_t)

            for i in range(n_s):
                for j in range(n_t):
                    index_matrix[i, n_s + j] = -1. / float(n_s)
                    index_matrix[n_s + j, i] = -1. / float(n_t)
        else:
            if n_s > 1:
                off_diag_s = 1.0 - torch.eye(n_s)
                index_matrix[:n_s, :n_s] = off_diag_s / float(n_s * (n_s - 1))
            if n_t > 1:
                off_diag_t = 1.0 - torch.eye(n_t)
                index_matrix[n_s:, n_s:] = off_diag_t / float(n_t * (n_t - 1))
            index_matrix[:n_s, n_s:] = -2. / float(n_s * n_t)
            index_matrix[n_s:, :n_s] = -2. / float(n_s * n_t)

        return index_matrix


def build_mmd_loss():
    return MultipleKernelMaximumMeanDiscrepancy([GaussianKernel(alpha=a) for a in MMD_ALPHAS])


class MLPClassifierHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout_p=DROPOUT_P):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        feat = self.feature_extractor(x)
        logits = self.fc(feat)
        return logits, feat


@torch.no_grad()
def extract_all_encoder_features(X_raw, batch_size=HF_BATCH_SIZE):
    """Encoder is frozen, so its output is fixed — compute once and reuse every round."""
    encoder.eval()
    feats = []
    for i in range(0, len(X_raw), batch_size):
        batch = torch.FloatTensor(X_raw[i:i + batch_size]).to(device)
        feats.append(encoder(batch))
    return torch.cat(feats, dim=0)


@torch.no_grad()
def predict_from_features(classifier, feats, batch_size=HF_BATCH_SIZE):
    classifier.eval()
    preds = []
    for i in range(0, feats.size(0), batch_size):
        logits, _ = classifier(feats[i:i + batch_size])
        preds.append(logits.argmax(dim=1))
    return torch.cat(preds, dim=0)


@torch.no_grad()
def build_correct_bank(classifier, feats, labels, hard_mask, num_classes, batch_size=HF_BATCH_SIZE):
    """Per-class hidden features of correctly classified samples (eval mode, detached).
    Classes with fewer than 2 correct samples are omitted."""
    classifier.eval()
    all_feat = []
    for i in range(0, feats.size(0), batch_size):
        _, feat_b = classifier(feats[i:i + batch_size])
        all_feat.append(feat_b)
    all_feat = torch.cat(all_feat, dim=0)
    classifier.train()

    correct = ~hard_mask
    bank = {}
    for c in range(num_classes):
        m = (labels == c) & correct
        if int(m.sum()) >= 2:
            bank[c] = all_feat[m]
    return bank


def class_conditional_mmd(hidden_feat, label_b, hard_b, bank, mmd_loss_fn, target_size):
    """Mean over classes of MK-MMD(hard samples of class c in the batch, correct samples of class c)."""
    terms = []
    for c in label_b[hard_b].unique():
        target = bank.get(int(c))
        if target is None:
            continue
        if target.size(0) > target_size:
            sel = torch.randperm(target.size(0), device=target.device)[:target_size]
            target = target[sel]
        z_s = hidden_feat[(label_b == c) & hard_b]
        terms.append(mmd_loss_fn(z_s, target))
    if not terms:
        return hidden_feat.new_zeros(())
    return torch.stack(terms).mean()


def train_classifier_weighted(classifier, optimizer, feats, labels, weights,
                               epochs, batch_size, hard_mask=None,
                               mmd_weight=0.0, mmd_loss_fn=None,
                               target_size=MMD_TARGET_SIZE):
    use_mmd = (mmd_loss_fn is not None and mmd_weight > 0
               and hard_mask is not None and bool(hard_mask.any()))
    hard_flags = hard_mask if hard_mask is not None else torch.zeros_like(labels, dtype=torch.bool)

    dataset = torch.utils.data.TensorDataset(feats, labels, weights, hard_flags)
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        drop_last=(len(dataset) > batch_size)
    )
    classifier.train()
    for epoch in range(epochs):
        bank = build_correct_bank(classifier, feats, labels, hard_flags, NUM_CLASSES) if use_mmd else None
        epoch_loss, epoch_ce, epoch_reg, n_batches = 0.0, 0.0, 0.0, 0
        for feat_b, label_b, weight_b, hard_b in loader:
            logits, hidden_feat = classifier(feat_b)
            per_sample_ce = F.cross_entropy(logits, label_b, reduction='none')
            ce_loss = (per_sample_ce * weight_b).mean()

            if use_mmd and bool(hard_b.any()):
                reg_loss = class_conditional_mmd(hidden_feat, label_b, hard_b, bank, mmd_loss_fn, target_size)
                loss = ce_loss + mmd_weight * reg_loss
            else:
                reg_loss = torch.tensor(0.0, device=feat_b.device)
                loss = ce_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_ce += ce_loss.item()
            epoch_reg += reg_loss.item()
            n_batches += 1
    return epoch_loss / n_batches, epoch_ce / n_batches, epoch_reg / n_batches


def run_iterative_hard_example_finetuning(X_raw, y):
    """Single-config run with the default constants (the tuning section below does not call this)."""
    print("\n" + "=" * 60)
    print("ITERATIVE HARD-EXAMPLE FINE-TUNING (encoder frozen, classifier-only)")
    print("=" * 60)

    N = len(y)
    y_tensor = torch.LongTensor(y).to(device)
    enc_feats_all = extract_all_encoder_features(X_raw)

    classifier = MLPClassifierHead(CLS_HIDDEN, CLS_HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(classifier.parameters(), lr=HF_LR, weight_decay=WEIGHT_DECAY)
    mmd_fn = build_mmd_loss()

    sample_weights = torch.full((N,), NORMAL_WEIGHT, device=device)
    hard_mask = torch.zeros(N, dtype=torch.bool, device=device)
    prev_n_correct = None

    for round_idx in range(1, HF_ROUNDS + 1):
        avg_loss, avg_ce, avg_reg = train_classifier_weighted(
            classifier, optimizer, enc_feats_all, y_tensor, sample_weights,
            epochs=HF_EPOCHS_PER_ROUND, batch_size=HF_BATCH_SIZE,
            hard_mask=hard_mask, mmd_weight=MMD_WEIGHT, mmd_loss_fn=mmd_fn
        )

        preds = predict_from_features(classifier, enc_feats_all)
        correct_mask = (preds == y_tensor)
        hard_mask = ~correct_mask
        n_correct = int(correct_mask.sum().item())
        n_hard = N - n_correct
        train_acc = n_correct / N

        sample_weights = torch.where(
            correct_mask,
            torch.full((N,), NORMAL_WEIGHT, device=device),
            torch.full((N,), HARD_WEIGHT, device=device),
        )

        delta_str = "" if prev_n_correct is None else f" (+{n_correct - prev_n_correct})"
        prev_n_correct = n_correct

        print(f"round {round_idx}/{HF_ROUNDS} | loss {avg_loss:.4f} (ce {avg_ce:.4f}, mmd {avg_reg:.4f}) | "
              f"correct {n_correct}/{N}{delta_str} | train_acc {train_acc:.4f} | "
              f"hard_examples {n_hard}/{N} ({n_hard / N:.2%})")

    return classifier, enc_feats_all


# ============================================================================
# HYPERPARAMETER TUNING FOR BOTH METHODS
# Random search per method, scored on a stratified validation split carved
# out of the training set (the test set is never used for selection). Each
# trial is averaged over TUNE_SEEDS seeds to reduce noise, since the gap
# between methods is small. After selection, both methods are retrained on
# the full training set with fresh seeds and evaluated once on the test set.
# ============================================================================
import json
from sklearn.model_selection import train_test_split

TUNE_TRIALS = 15        # random-search trials per method (same budget for both)
TUNE_SEEDS = 2          # seeds averaged per trial
VAL_FRACTION = 0.2
FINAL_SEEDS = 3         # seeds for the final test evaluation
TUNE_SEED_BASE = 0
FINAL_SEED_BASE = 100   # disjoint from tuning seeds

BASELINE_SPACE = {
    'lr': [3e-4, 5e-4, 1e-3, 2e-3],
    'weight_decay': [0.0, 1e-5, 1e-4, 1e-3],
    'dropout': [0.0, 0.1, 0.3, 0.5],
    'epochs': [10, 20, 30, 50],
    'batch_size': [32, 64, 128],
}

HARD_SPACE = {
    'lr': [3e-4, 5e-4, 1e-3, 2e-3],
    'weight_decay': [0.0, 1e-5, 1e-4, 1e-3],
    'dropout': [0.0, 0.1, 0.3, 0.5],
    'rounds': [5, 10, 15],
    'epochs_per_round': [2, 3, 5],
    'hard_weight': [2.0, 3.0, 5.0, 8.0],
    'mmd_weight': [0.0, 0.05, 0.1, 0.5, 1.0],   # 0.0 disables the MK-MMD term
    'batch_size': [32, 64, 128],
}


def sample_trials(space, n_trials, seed=0):
    rng = np.random.default_rng(seed)
    trials, seen = [], set()
    max_unique = int(np.prod([len(v) for v in space.values()]))
    n_trials = min(n_trials, max_unique)
    while len(trials) < n_trials:
        cfg = {k: v[rng.integers(len(v))] for k, v in space.items()}
        key = tuple(cfg.items())
        if key in seen:
            continue
        seen.add(key)
        trials.append(cfg)
    return trials


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def fit_baseline(feats, labels, cfg, seed):
    set_seed(seed)
    clf = MLPClassifierHead(CLS_HIDDEN, CLS_HIDDEN, NUM_CLASSES, dropout_p=cfg['dropout']).to(device)
    opt = optim.Adam(clf.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    weights = torch.full((len(labels),), NORMAL_WEIGHT, device=device)
    train_classifier_weighted(clf, opt, feats, labels, weights,
                               epochs=cfg['epochs'], batch_size=cfg['batch_size'])
    return clf


def fit_hard_example(feats, labels, cfg, seed):
    set_seed(seed)
    clf = MLPClassifierHead(CLS_HIDDEN, CLS_HIDDEN, NUM_CLASSES, dropout_p=cfg['dropout']).to(device)
    opt = optim.Adam(clf.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    weights = torch.full((len(labels),), NORMAL_WEIGHT, device=device)
    hard_mask = torch.zeros(len(labels), dtype=torch.bool, device=device)
    mmd_fn = build_mmd_loss()

    for _ in range(cfg['rounds']):
        train_classifier_weighted(
            clf, opt, feats, labels, weights,
            epochs=cfg['epochs_per_round'], batch_size=cfg['batch_size'],
            hard_mask=hard_mask, mmd_weight=cfg['mmd_weight'], mmd_loss_fn=mmd_fn
        )
        correct = predict_from_features(clf, feats) == labels
        hard_mask = ~correct
        weights = torch.where(
            correct,
            torch.full_like(weights, NORMAL_WEIGHT),
            torch.full_like(weights, cfg['hard_weight']),
        )
    return clf


def accuracy_on(clf, feats, labels):
    return (predict_from_features(clf, feats) == labels).float().mean().item()


def tune_method(name, fit_fn, space, feats_tr, y_tr, feats_val, y_val):
    print("\n" + "=" * 60)
    print(f"TUNING: {name} ({TUNE_TRIALS} trials x {TUNE_SEEDS} seeds)")
    print("=" * 60)

    rows, best_cfg, best_acc = [], None, -1.0
    for t, cfg in enumerate(sample_trials(space, TUNE_TRIALS, seed=1), start=1):
        accs = []
        for s in range(TUNE_SEEDS):
            clf = fit_fn(feats_tr, y_tr, cfg, TUNE_SEED_BASE + s)
            accs.append(accuracy_on(clf, feats_val, y_val))
        mean_acc = float(np.mean(accs))
        rows.append({**cfg, 'val_acc': mean_acc, 'val_std': float(np.std(accs))})
        if mean_acc > best_acc:
            best_acc, best_cfg = mean_acc, cfg
        print(f"trial {t}/{TUNE_TRIALS} | val_acc {mean_acc:.4f} (best {best_acc:.4f}) | {cfg}")

    results = pd.DataFrame(rows).sort_values('val_acc', ascending=False)
    print(f"\nBest {name} config (val_acc {best_acc:.4f}): {best_cfg}")
    return best_cfg, best_acc, results


# ---- stratified train/val split on the fixed encoder features ----
X_train_enc_feats = extract_all_encoder_features(X_train_raw)
y_train_tensor = torch.LongTensor(y_train).to(device)

idx_tr, idx_val = train_test_split(
    np.arange(len(y_train)), test_size=VAL_FRACTION, stratify=y_train, random_state=42
)
idx_tr_t = torch.LongTensor(idx_tr).to(device)
idx_val_t = torch.LongTensor(idx_val).to(device)
feats_tr, y_tr = X_train_enc_feats[idx_tr_t], y_train_tensor[idx_tr_t]
feats_val, y_val = X_train_enc_feats[idx_val_t], y_train_tensor[idx_val_t]
print(f"Tuning split: train {len(idx_tr)} / val {len(idx_val)}")

best_baseline_cfg, best_baseline_val, baseline_results = tune_method(
    'UNIFORM-WEIGHT BASELINE', fit_baseline, BASELINE_SPACE, feats_tr, y_tr, feats_val, y_val
)
best_hard_cfg, best_hard_val, hard_results = tune_method(
    'HARD-EXAMPLE + MK-MMD', fit_hard_example, HARD_SPACE, feats_tr, y_tr, feats_val, y_val
)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
baseline_results.to_csv(os.path.join(OUTPUT_FOLDER, 'tuning_baseline.csv'), index=False)
hard_results.to_csv(os.path.join(OUTPUT_FOLDER, 'tuning_hard_example.csv'), index=False)
with open(os.path.join(OUTPUT_FOLDER, 'best_params.json'), 'w') as f:
    json.dump({'baseline': {**best_baseline_cfg, 'val_acc': best_baseline_val},
               'hard_example': {**best_hard_cfg, 'val_acc': best_hard_val}}, f, indent=2)


# ============================================================================
# FINAL EVALUATION: best configs retrained on the full training set,
# evaluated once on the test set, averaged over FINAL_SEEDS seeds
# ============================================================================

X_test_emb = extract_embeddings_from_raw(emb_model, X_test_raw)
X_test_enc_feats = torch.FloatTensor(X_test_emb).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

unique_labels = np.unique(np.concatenate([y_train, y_test]))
target_names = [label_names.get(label, f'Class_{label}') for label in unique_labels]


def print_and_save_report(y_true, y_pred, tag, filename):
    print(f"\n{'='*60}")
    print(f"CLASSIFICATION REPORT — {tag}")
    print(f"{'='*60}\n")
    report = classification_report(
        y_true, y_pred,
        target_names=[str(x) for x in target_names],
        digits=4, zero_division=0
    )
    print(report)

    report_path = os.path.join(OUTPUT_FOLDER, filename)
    with open(report_path, 'w') as f:
        f.write(f"CLASSIFICATION REPORT — {tag}\n")
        f.write("="*60 + "\n\n")
        f.write(report)
    print(f"\nReport saved to: {report_path}")
    return report


def final_eval(name, fit_fn, cfg, report_file):
    accs, first_pred = [], None
    for s in range(FINAL_SEEDS):
        clf = fit_fn(X_train_enc_feats, y_train_tensor, cfg, FINAL_SEED_BASE + s)
        accs.append(accuracy_on(clf, X_test_enc_feats, y_test_tensor))
        if first_pred is None:
            first_pred = predict_from_features(clf, X_test_enc_feats).cpu().numpy()
        print(f"  {name} seed {s + 1}/{FINAL_SEEDS} | test_acc {accs[-1]:.4f}")
    print_and_save_report(y_test, first_pred, f'{name} (best config, seed 1)', report_file)
    return float(np.mean(accs)), float(np.std(accs))


mean_base, std_base = final_eval('UNIFORM-WEIGHT BASELINE', fit_baseline, best_baseline_cfg,
                                  'classification_report_baseline_tuned.txt')
mean_hard, std_hard = final_eval('HARD-EXAMPLE + MK-MMD', fit_hard_example, best_hard_cfg,
                                  'classification_report_hard_example_tuned.txt')

print(f"\n{'='*60}")
print(f"Test accuracy (mean +/- std over {FINAL_SEEDS} seeds), tuned configs")
print(f"  uniform-weight baseline : {mean_base:.4f} +/- {std_base:.4f}   (val {best_baseline_val:.4f})")
print(f"  hard-example + mk-mmd  : {mean_hard:.4f} +/- {std_hard:.4f}   (val {best_hard_val:.4f})")
print(f"{'='*60}")

Tuning split: train 8000 / val 2000

TUNING: UNIFORM-WEIGHT BASELINE (15 trials x 2 seeds)
trial 1/15 | val_acc 0.9958 (best 0.9958) | {'lr': 0.0005, 'weight_decay': 0.0001, 'dropout': 0.5, 'epochs': 50, 'batch_size': 32}
trial 2/15 | val_acc 0.9965 (best 0.9965) | {'lr': 0.0003, 'weight_decay': 0.001, 'dropout': 0.5, 'epochs': 10, 'batch_size': 32}
trial 3/15 | val_acc 0.9950 (best 0.9965) | {'lr': 0.002, 'weight_decay': 1e-05, 'dropout': 0.1, 'epochs': 50, 'batch_size': 32}
trial 4/15 | val_acc 0.9968 (best 0.9968) | {'lr': 0.0005, 'weight_decay': 0.0001, 'dropout': 0.3, 'epochs': 10, 'batch_size': 32}
trial 5/15 | val_acc 0.9953 (best 0.9968) | {'lr': 0.002, 'weight_decay': 0.001, 'dropout': 0.5, 'epochs': 30, 'batch_size': 128}
trial 6/15 | val_acc 0.9965 (best 0.9968) | {'lr': 0.0005, 'weight_decay': 1e-05, 'dropout': 0.5, 'epochs': 10, 'batch_size': 32}
trial 7/15 | val_acc 0.9965 (best 0.9968) | {'lr': 0.0003, 'weight_decay': 1e-05, 'dropout': 0.5, 'epochs': 10, 'batch_size': 64